# Servidor de inferencia Peaje IA — RunPod **o** Google Colab
Levanta el **mismo** `inferencia/app.py` (M1 placas + M2 vehículos + M3 ejes + M4 pasajeros).
Laravel solo cambia `INFERENCE_URL` en *Panel → Estado del sistema*:

| Plataforma | URL pública | Cómo se expone |
|---|---|---|
| **RunPod** (principal, bono +5) | `https://<POD_ID>-8000.proxy.runpod.net` | *Expose HTTP Ports: 8000* |
| **Colab** (respaldo) | `https://xxxx.trycloudflare.com` | túnel Cloudflare (sin cuenta) |

El servidor escucha en `0.0.0.0:8000` (con `127.0.0.1` el proxy de RunPod responde 502).

## 1. Configuración

In [ ]:
import os, secrets, sys
REPO_URL = 'https://github.com/jayutomas623/sistema-peaje-ia.git'   # <- su repositorio público
PUERTO = 8000
EN_COLAB = 'google.colab' in sys.modules or 'COLAB_RELEASE_TAG' in os.environ
POD_ID = os.environ.get('RUNPOD_POD_ID')
BASE = '/content' if EN_COLAB else '/workspace'

# Token compartido con Laravel. En RunPod defínalo como variable de entorno API_TOKEN del Pod.
os.environ['API_TOKEN'] = os.environ.get('API_TOKEN') or secrets.token_urlsafe(32)
print('Plataforma:', 'Colab' if EN_COLAB else ('RunPod ' + POD_ID if POD_ID else 'otra'))
print('API_TOKEN :', os.environ['API_TOKEN'])

## 2. Código e instalación (versiones fijadas de `requirements-venv.txt`)

In [ ]:
DIR_REPO = f'{BASE}/sistema-peaje-ia'
if not os.path.exists(DIR_REPO):
    !git clone -q --depth 1 {REPO_URL} {DIR_REPO}
else:
    !cd {DIR_REPO} && git pull -q
DIR_INF = f'{DIR_REPO}/inferencia'
%cd {DIR_INF}
!apt-get -qq install -y libgl1 > /dev/null 2>&1 || true
if EN_COLAB:
    # Colab ya trae torch con CUDA: se instala el resto sin reemplazarlo
    !grep -vE "^(torch|torchvision)==" requirements-venv.txt > /tmp/req.txt && pip -q install -r /tmp/req.txt
else:
    !pip -q install -r requirements-venv.txt

## 3. Pesos entrenados (M2, M3, M4)
Copie `m2_vehiculos_seg.pt`, `m3_llantas_pose.pt` y `m4_pasajeros_seg.pt` a `inferencia/pesos/`.
* **Colab**: se copian desde `MyDrive/peaje-ia/pesos/`.
* **RunPod**: súbalos con el botón *Upload* de JupyterLab o `runpodctl send`.
Si faltan, el servidor arranca en **modo fallback** (preentrenados COCO / YOLOE) y lo avisa en `/health`.

In [ ]:
if EN_COLAB:
    try:
        from google.colab import drive; drive.mount('/content/drive')
        !cp -v /content/drive/MyDrive/peaje-ia/pesos/*.pt {DIR_INF}/pesos/ 2>/dev/null || echo "Sin pesos en Drive"
    except Exception as e:
        print(e)
!ls -la {DIR_INF}/pesos

## 4. Diagnóstico del entorno (evidencia para el bono de entornos)

In [ ]:
!python diagnostico_entorno.py --nombre {'colab' if EN_COLAB else 'runpod'} --salida evidencias

## 5. Iniciar el servidor (hilo en segundo plano)

In [ ]:
import threading, time, uvicorn
sys.path.insert(0, DIR_INF)
import app as servidor_app

if 'servidor' in globals() and servidor.started and not servidor.should_exit:
    print('El servidor ya está corriendo.')
else:
    servidor = uvicorn.Server(uvicorn.Config(servidor_app.app, host='0.0.0.0', port=PUERTO, log_level='warning'))
    hilo = threading.Thread(target=servidor.run, daemon=True); hilo.start()
    for _ in range(600):              # la carga de modelos tarda 20-60 s la primera vez
        if servidor.started: break
        time.sleep(0.5)
    print('Servidor iniciado:', servidor.started)

## 6. URL pública

In [ ]:
import re, subprocess
if POD_ID:
    URL_PUBLICA = f'https://{POD_ID}-{PUERTO}.proxy.runpod.net'
elif EN_COLAB:
    !wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod +x /usr/local/bin/cloudflared
    tunel = subprocess.Popen(['cloudflared', 'tunnel', '--no-autoupdate', '--url', f'http://localhost:{PUERTO}'],
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    log, URL_PUBLICA = [], None
    threading.Thread(target=lambda: [log.append(l) for l in tunel.stdout], daemon=True).start()
    for _ in range(60):
        m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', ''.join(log))
        if m: URL_PUBLICA = m.group(0); break
        time.sleep(0.5)
    time.sleep(3)
else:
    URL_PUBLICA = f'http://<IP-PUBLICA>:{PUERTO}'
print('=' * 70)
print('INFERENCE_URL   =', URL_PUBLICA)
print('INFERENCE_TOKEN =', os.environ['API_TOKEN'])
print('Péguelos en Laravel > Estado del sistema (sin redeploy).')
print('=' * 70)

## 7. Prueba (health + inferencia con una foto boliviana del repo)

In [ ]:
import json, glob, base64, urllib.request
from IPython.display import Image, display
print(json.dumps(json.loads(urllib.request.urlopen(f'{URL_PUBLICA}/health', timeout=60).read()), indent=1)[:1500])
foto = sorted(glob.glob(f'{DIR_REPO}/web/database/seeders/evidencias/*.jpg'))[0]
req = urllib.request.Request(f'{URL_PUBLICA}/inferir', data=open(foto, 'rb').read(), method='POST',
                             headers={'Content-Type': 'image/jpeg', 'X-API-Token': os.environ['API_TOKEN']})
res = json.loads(urllib.request.urlopen(req, timeout=120).read())
img64 = res.pop('imagen_anotada_base64')
print(json.dumps(res, indent=1, ensure_ascii=False))
display(Image(data=base64.b64decode(img64.split(',', 1)[1]), width=500))

## 8. Detener (solo al terminar la demo)

In [ ]:
DETENER = False
if DETENER:
    servidor.should_exit = True
    if 'tunel' in globals(): tunel.terminate()
    print('Servidor detenido')